# CALVIN Dynamics World Model Evaluation

Loads a trained CALVIN dynamics world model and compares one-step next-state transition error against a zero-delta predictor on the validation split saved in the run provenance.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "calvin_experiments" / "train_dynamics_world_model.py").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("Could not find repository root")
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from calvin_experiments.train_dynamics_world_model import (
    build_calvin_dynamics_trajectories,
    flatten_trajectories,
    load_dynamics_model_for_eval,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
# Set RUN_PATH to either a run directory or a checkpoint file.
# If left as None, the newest run under calvin/dynamics_world_model is used.
RUN_PATH = None
CHECKPOINT_NAME = "best_model.pt"
SPLIT = "val"  # "val", "train", or "all"
BATCH_SIZE = 8192

if RUN_PATH is None:
    run_root = REPO_ROOT / "calvin" / "dynamics_world_model"
    runs = sorted([p for p in run_root.glob("*") if p.is_dir()], key=lambda p: p.stat().st_mtime)
    if not runs:
        raise FileNotFoundError(f"No dynamics world model runs found under {run_root}")
    RUN_PATH = runs[-1]
else:
    RUN_PATH = Path(RUN_PATH)
    if not RUN_PATH.is_absolute():
        RUN_PATH = REPO_ROOT / RUN_PATH

model, stats, checkpoint, meta = load_dynamics_model_for_eval(
    RUN_PATH,
    device=device,
    checkpoint_name=CHECKPOINT_NAME,
)
run_dir = Path(meta["run_dir"])
provenance = json.loads((run_dir / "data_provenance.json").read_text())

print("run_dir:", run_dir)
print("checkpoint:", meta["checkpoint_path"])
print("checkpoint epoch:", checkpoint.get("epoch"))
print("checkpoint val_loss:", checkpoint.get("val_loss"))
print("model_config:", checkpoint["model_config"])
print("device:", device)

In [ ]:
dataset_path = Path(provenance["dataset"])
if not dataset_path.is_absolute():
    dataset_path = REPO_ROOT / dataset_path

if SPLIT == "train":
    demo_refs = provenance["train_demos"]
elif SPLIT == "val":
    demo_refs = provenance["val_demos"]
elif SPLIT == "all":
    demo_refs = provenance["train_demos"] + provenance["val_demos"]
else:
    raise ValueError(f"Unknown split: {SPLIT}")

demo_keys = [demo["demo_id"] for demo in demo_refs]
trajectories = build_calvin_dynamics_trajectories(dataset_path, demo_keys)
flat = flatten_trajectories(trajectories)

print("dataset:", dataset_path)
print("split:", SPLIT)
print("demos:", len(trajectories))
print("transitions:", flat["states"].shape[0])
print("state_dim:", flat["states"].shape[1])
print("action_dim:", flat["actions"].shape[1])

In [ ]:
def predict_deltas_raw(model, flat, stats, batch_size, device):
    states_n = ((flat["states"] - stats["state_mean"]) / stats["state_std"]).astype(np.float32)
    actions_n = ((flat["actions"] - stats["action_mean"]) / stats["action_std"]).astype(np.float32)
    preds = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(states_n), batch_size):
            end = start + batch_size
            pred_n = model(
                torch.from_numpy(states_n[start:end]).to(device),
                torch.from_numpy(actions_n[start:end]).to(device),
            )
            preds.append(pred_n.cpu().numpy())
    pred_n = np.concatenate(preds, axis=0)
    return pred_n * stats["delta_std"] + stats["delta_mean"]


def summarize_transition_errors(name, pred_next, true_next):
    err = pred_next - true_next
    l2 = np.linalg.norm(err, axis=-1)
    return {
        "name": name,
        "mse_per_value": float(np.mean(np.square(err))),
        "rmse_per_value": float(np.sqrt(np.mean(np.square(err)))),
        "mae_per_value": float(np.mean(np.abs(err))),
        "max_abs_per_value": float(np.max(np.abs(err))),
        "l2_mean": float(np.mean(l2)),
        "l2_median": float(np.median(l2)),
        "l2_p95": float(np.percentile(l2, 95)),
        "l2_max": float(np.max(l2)),
    }


pred_delta = predict_deltas_raw(model, flat, stats, BATCH_SIZE, device)
pred_next = flat["states"] + pred_delta
zero_next = flat["states"]
true_next = flat["next_states"]

learned_summary = summarize_transition_errors("learned", pred_next, true_next)
zero_summary = summarize_transition_errors("zero_delta", zero_next, true_next)

print(f"{'model':<14} {'rmse/value':>12} {'mae/value':>12} {'l2 median':>12} {'l2 p95':>12} {'max abs':>12}")
print("-" * 78)
for row in [learned_summary, zero_summary]:
    print(
        f"{row['name']:<14} {row['rmse_per_value']:12.6f} {row['mae_per_value']:12.6f} "
        f"{row['l2_median']:12.6f} {row['l2_p95']:12.6f} {row['max_abs_per_value']:12.6f}"
    )

improvement = 1.0 - learned_summary["mse_per_value"] / max(zero_summary["mse_per_value"], 1e-12)
print(f"\nMSE improvement over zero-delta predictor: {100.0 * improvement:.2f}%")

In [ ]:
learned_l2 = np.linalg.norm(pred_next - true_next, axis=-1)
zero_l2 = np.linalg.norm(zero_next - true_next, axis=-1)

fig, ax = plt.subplots(figsize=(8, 5))
for label, values in [("learned", learned_l2), ("zero_delta", zero_l2)]:
    xs = np.sort(values)
    ys = np.linspace(0.0, 1.0, len(xs), endpoint=True)
    ax.plot(xs, ys, label=label, linewidth=2)
ax.set_xlabel("one-step next-state L2 error")
ax.set_ylabel("CDF")
ax.set_title(f"CALVIN dynamics transition error ({SPLIT}, n={len(learned_l2)})")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
report = {
    "run_dir": str(run_dir),
    "checkpoint_path": meta["checkpoint_path"],
    "checkpoint_epoch": int(checkpoint.get("epoch", -1)),
    "checkpoint_val_loss": None if checkpoint.get("val_loss") is None else float(checkpoint["val_loss"]),
    "dataset": str(dataset_path),
    "split": SPLIT,
    "num_demos": int(len(trajectories)),
    "num_transitions": int(flat["states"].shape[0]),
    "learned": learned_summary,
    "zero_delta": zero_summary,
    "mse_improvement_over_zero_delta": float(improvement),
}

report_path = run_dir / f"{CHECKPOINT_NAME.replace('.pt', '')}_{SPLIT}_transition_error_report.json"
report_path.write_text(json.dumps(report, indent=2))
print("saved:", report_path)